In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# First cell: Setup environment
!pip install chess stockfish tqdm

# Verify files (update paths to match your dataset names)
!ls /kaggle/input/nnue-tr

!cp /kaggle/input/nnue-tr/stockfish_linux_x64 /kaggle/working/
!cp /kaggle/input/stockfish/stockfish-windows-x86-64-avx2.exe /kaggle/working/
!cp /kaggle/input/stockfish/stockfish-windows-x86-64-avx2.exe /kaggle/working/
!ls /kaggle/working

In [ ]:
import sys
import chess
import chess.pgn
import stockfish
import csv
from tqdm import tqdm
!chmod +x /kaggle/working/stockfish_linux_x64

# --- Configuration ---
STOCKFISH_PATH     = "/kaggle/working/stockfish_linux_x64"
PGN_FILE_PATH      = "/kaggle/input/nnue-tr/games.pgn"
OUTPUT_CSV_PATH    = "training_data.csv"
EVALUATION_DEPTH   = 12   # Stockfish depth
MAX_POSITIONS      = 100000

# --- Initialization ---
try:
    sf = stockfish.Stockfish(path=STOCKFISH_PATH, depth=EVALUATION_DEPTH)
    print(f"Stockfish version: {sf.get_stockfish_major_version()}")
except Exception as e:
    print(f"Error initializing Stockfish: {e}")
    print(f"Please ensure the path '{STOCKFISH_PATH}' is correct and executable.")
    sys.exit(1)

def generate_training_data():
    total_positions = 0
    total_games = 0

    with open(PGN_FILE_PATH) as pgn_file, \
         open(OUTPUT_CSV_PATH, 'w', newline='') as csvfile:

        writer = csv.writer(csvfile)
        writer.writerow(['fen', 'evaluation', 'result'])

        # Stream through games until we hit the position limit
        while total_positions < MAX_POSITIONS:
            game = chess.pgn.read_game(pgn_file)
            if game is None:
                break  # end of file

            total_games += 1
            result = game.headers.get("Result", "*")
            board  = game.board()

            # iterate moves with a progress bar per game
            for move in tqdm(game.mainline_moves(),
                              desc=f"Game {total_games} ({game.headers.get('Site','?')})",
                              leave=False):

                board.push(move)
                fen = board.fen()

                # ask Stockfish
                sf.set_fen_position(fen)
                ev = sf.get_evaluation()

                # convert to centipawns
                if ev['type'] == 'cp':
                    score = ev['value']
                elif ev['type'] == 'mate':
                    m = ev['value']
                    score = (30000 - abs(m)) * (1 if m > 0 else -1)
                else:
                    continue

                # always from White’s perspective
                if not board.turn:
                    score = -score

                writer.writerow([fen, score, result])
                total_positions += 1

                if total_positions >= MAX_POSITIONS:
                    break

        print(f"\nProcessing complete.")
        print(f"Total positions generated: {total_positions}")
        print(f"Total games processed: {total_games}")
        if total_positions >= MAX_POSITIONS:
            print(f"(Stopped at {total_positions} positions, mid-game on game #{total_games}.)")
        else:
            print("(Reached end of PGN file.)")

if __name__ == "__main__":
    generate_training_data()


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import chess

# --- Configuration ---
INPUT_DIM = 768  # 64 squares * 12 piece types (6 white, 6 black)
HIDDEN_DIM = 256
OUTPUT_DIM = 1    # A single evaluation score
LEARNING_RATE = 0.001
EPOCHS = 20
BATCH_SIZE = 1024
DATA_PATH = "/kaggle/working/training_data.csv"
MODEL_SAVE_PATH = "nnue_model.pth"

# --- Helper Function: FEN to Tensor ---
def fen_to_input_tensor(fen):
    """
    Converts a FEN string to a flat tensor for the network input.
    This is a simplified representation. A real NNUE uses a 'half-KP' sparse representation.
    """
    board = chess.Board(fen)
    tensor = torch.zeros(INPUT_DIM)
    piece_map = board.piece_map()
    for square, piece in piece_map.items():
        piece_idx = piece.piece_type - 1 + (6 if piece.color == chess.BLACK else 0)
        tensor[square * 12 + piece_idx] = 1.0
    return tensor

# --- PyTorch Dataset ---
class ChessDataset(Dataset):
    def __init__(self, csv_file):
        self.data = pd.read_csv(csv_file)
        # Clamp extreme evaluations for stability
        self.data['evaluation'] = self.data['evaluation'].clip(-1500, 1500)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data.iloc[idx]
        fen = item['fen']
        # Convert FEN to our input tensor format
        input_tensor = fen_to_input_tensor(fen)
        # Evaluation is the target
        evaluation = torch.tensor([item['evaluation']], dtype=torch.float32)
        return input_tensor, evaluation

# --- NNUE Model Architecture ---
class NNUE(nn.Module):
    def __init__(self):
        super(NNUE, self).__init__()
        self.input_layer = nn.Linear(INPUT_DIM, HIDDEN_DIM)
        self.hidden_layer = nn.Linear(HIDDEN_DIM, 32)
        self.output_layer = nn.Linear(32, OUTPUT_DIM)
    
    def forward(self, x):
        # Activation function (ReLU) is applied after linear layers
        x = torch.relu(self.input_layer(x))
        x = torch.relu(self.hidden_layer(x))
        x = self.output_layer(x)
        return x

# --- Training Loop ---
def train_model():
    # Set device to GPU if available, otherwise CPU
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Training on {device}")

    # Load dataset
    dataset = ChessDataset(DATA_PATH)
    train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)

    # Initialize model, optimizer, and loss function
    model = NNUE().to(device)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.MSELoss() # Mean Squared Error is common for regression tasks

    print("Starting training...")
    for epoch in range(EPOCHS):
        total_loss = 0.0
        for i, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)

            # Zero the parameter gradients
            optimizer.zero_grad()

            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            # Backward pass and optimize
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            if (i + 1) % 50 == 0:
                print(f'Epoch [{epoch+1}/{EPOCHS}], Step [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}')
        
        print(f"--- Epoch {epoch+1} finished. Average Loss: {total_loss / len(train_loader):.4f} ---")

    # Save the trained model's state
    torch.save(model.state_dict(), MODEL_SAVE_PATH)
    print(f"Training finished. Model saved to {MODEL_SAVE_PATH}")


if __name__ == "__main__":
    train_model()

In [ ]:
import torch
import struct

# --- Configuration ---
INPUT_DIM = 768  # 64 squares * 12 piece types (6 white, 6 black)
HIDDEN_DIM = 256
MODEL_SAVE_PATH = "/kaggle/working/nnue_model.pth"
class NNUE(nn.Module):
    def __init__(self):
        super(NNUE, self).__init__()
        self.input_layer = nn.Linear(INPUT_DIM, HIDDEN_DIM)
        self.hidden_layer = nn.Linear(HIDDEN_DIM, 32)
        self.output_layer = nn.Linear(32, OUTPUT_DIM)
    
    def forward(self, x):
        # Activation function (ReLU) is applied after linear layers
        x = torch.relu(self.input_layer(x))
        x = torch.relu(self.hidden_layer(x))
        x = self.output_layer(x)
        return x

# --- Configuration ---
BINARY_OUTPUT_PATH = "nnue_model.bin"

def export_model():
    """
    Reads the trained PyTorch model and writes its weights
    and biases to a simple binary file.
    
    NOTE: The binary format MUST match what your C++ engine expects.
    This is a simplified example.
    """
    model = NNUE()
    model.load_state_dict(torch.load(MODEL_SAVE_PATH))
    model.eval() # Set model to evaluation mode

    with open(BINARY_OUTPUT_PATH, 'wb') as f:
        print("Writing weights and biases to binary file...")

        # 1. Input Layer
        # Write weights (matrix)
        weights = model.input_layer.weight.detach().numpy().flatten()
        f.write(struct.pack('f' * len(weights), *weights))
        # Write biases (vector)
        biases = model.input_layer.bias.detach().numpy().flatten()
        f.write(struct.pack('f' * len(biases), *biases))

        # 2. Hidden Layer
        weights = model.hidden_layer.weight.detach().numpy().flatten()
        f.write(struct.pack('f' * len(weights), *weights))
        biases = model.hidden_layer.bias.detach().numpy().flatten()
        f.write(struct.pack('f' * len(biases), *biases))
        
        # 3. Output Layer
        weights = model.output_layer.weight.detach().numpy().flatten()
        f.write(struct.pack('f' * len(weights), *weights))
        biases = model.output_layer.bias.detach().numpy().flatten()
        f.write(struct.pack('f' * len(biases), *biases))
        
    print(f"Model successfully exported to {BINARY_OUTPUT_PATH}")
    print("You can now load this file in your C++ chess engine.")

if __name__ == "__main__":
    export_model()